In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"]        = sys.executable

os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
print("go")

go


In [2]:
import os
import sys

# ✅ MUST come before any PySpark import
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pyspark
os.environ["SPARK_HOME"] = pyspark.__path__[0]

print("Python :", sys.executable)
print("SPARK_HOME :", os.environ["SPARK_HOME"])
print("go")

Python : c:\Users\nmira\AppData\Local\Programs\Python\Python311\python.exe
SPARK_HOME : C:\Users\nmira\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark
go


In [3]:
# Just add this at the top of your Colab notebook
!pip install pyspark -q


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Config Kawtar (legacy quick bootstrap - disabled to avoid starting Spark with wrong settings)
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.master("local[*]").appName("TaaSim").getOrCreate()
# print("go")

print("Legacy Spark bootstrap disabled. Run the next configuration cell.")

Legacy Spark bootstrap disabled. Run the next configuration cell.


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, LongType
import os
import sys

# Active Config (MyVersion style)
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
preferred_python = r"C:\Users\nmira\AppData\Local\Programs\Python\Python311\python.exe"
python_path = preferred_python if os.path.exists(preferred_python) else sys.executable
os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Restarting the session ensures updated Spark settings are applied.
try:
    spark.stop()
except Exception:
    pass

builder = SparkSession.builder \
    .appName("Taasim-Porto") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")

# Config Kawtar (MinIO + s3a). Uncomment if you want to read s3a:// paths.
# builder = builder \
#     .config("spark.jars.packages",
#             "org.apache.hadoop:hadoop-aws:3.3.4,"
#             "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
#     .config("spark.jars.repositories", "https://repo1.maven.org/maven2") \
#     .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
#     .config("spark.hadoop.fs.s3a.access.key", "admin") \
#     .config("spark.hadoop.fs.s3a.secret.key", "password123") \
#     .config("spark.hadoop.fs.s3a.path.style.access", "true") \
#     .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
#     .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
#     .config("spark.hadoop.fs.s3a.aws.credentials.provider",
#             "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)
print("Python:", python_path)
print("go")

Spark: 4.1.1
Python: C:\Users\nmira\AppData\Local\Programs\Python\Python311\python.exe
go


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType,
    BooleanType, LongType
)
from pyspark.sql.functions import col, count, when, isnan

porto_schema = StructType([
    StructField("TRIP_ID",      StringType(),  True),
    StructField("CALL_TYPE",    StringType(),  True),
    StructField("ORIGIN_CALL",  IntegerType(), True),
    StructField("ORIGIN_STAND", IntegerType(), True),
    StructField("TAXI_ID",      IntegerType(), True),
    StructField("TIMESTAMP",    LongType(),    True),
    StructField("DAY_TYPE",     StringType(),  True),
    StructField("MISSING_DATA", BooleanType(), True),
    StructField("POLYLINE",     StringType(),  True),
])

# Active source (MyVersion style)
porto_source = "train.csv"

# Config Kawtar source (uncomment if using MinIO + s3a config above)
# porto_source = "s3a://raw/porto-trips/train.csv"

print(f"Reading Porto source: {porto_source}")
df_raw = spark.read.csv(
    porto_source,
    header=True,
    schema=porto_schema
)

df_raw.printSchema()
df_raw.show(5, truncate=50)
print("go")

Reading Porto source: train.csv
root
 |-- TRIP_ID: string (nullable = true)
 |-- CALL_TYPE: string (nullable = true)
 |-- ORIGIN_CALL: integer (nullable = true)
 |-- ORIGIN_STAND: integer (nullable = true)
 |-- TAXI_ID: integer (nullable = true)
 |-- TIMESTAMP: long (nullable = true)
 |-- DAY_TYPE: string (nullable = true)
 |-- MISSING_DATA: boolean (nullable = true)
 |-- POLYLINE: string (nullable = true)

+-------------------+---------+-----------+------------+--------+----------+--------+------------+--------------------------------------------------+
|            TRIP_ID|CALL_TYPE|ORIGIN_CALL|ORIGIN_STAND| TAXI_ID| TIMESTAMP|DAY_TYPE|MISSING_DATA|                                          POLYLINE|
+-------------------+---------+-----------+------------+--------+----------+--------+------------+--------------------------------------------------+
|1372636858620000589|        C|       NULL|        NULL|20000589|1372636858|       A|       false|[[-8.618643,41.141412],[-8.618499,41.1413

## 1. Spark Session and Porto Data Ingestion

This cell initializes Spark, applies Windows-specific PySpark environment settings, and loads the Porto taxi dataset using an explicit schema. Defining the schema up front improves type consistency and avoids expensive schema inference on large CSV files.

In [7]:
from pyspark.sql.functions import col, count, when, isnan, min as spark_min, max as spark_max, from_unixtime, countDistinct

print("GENERAL DATA INSIGHTS : ")
#runcate=False so Spark doesn't hide the massive POLYLINE string
df_raw.select("TRIP_ID", "CALL_TYPE", "TAXI_ID", "TIMESTAMP", "MISSING_DATA", "POLYLINE").show(5, truncate=False)
print("Checking for NULL values")
df_raw.select([count(when(col(c).isNull(), c)).alias(c) for c in df_raw.columns]).show()
print("-" * 30)
df_clean=df_raw.filter(col("MISSING_DATA")== False)
clean_count=df_clean.count()
print(f"Valid rows remaining: {clean_count}")
print(f"Dropped {1710670 - clean_count} invalid trips.")
print("\n4. DATASET SANITY CHECK")
print("-" * 30)
df_clean.select( #from_unixtime  to convert timestamp to human readable date
    countDistinct("TAXI_ID").alias("Unique_Taxis"),
    from_unixtime(spark_min("TIMESTAMP")).alias("First_Trip_Date"),
    from_unixtime(spark_max("TIMESTAMP")).alias("Last_Trip_Date")

).show(truncate=False)#truncate=False to prevent spark abrivating dara

print("\n5. TIMESTAMP COMPARISON")
print("-" * 30)
# We select the raw TIMESTAMP, and then select it again but wrapped in from_unixtime()
df_clean.select(
    "TRIP_ID", 
    "TIMESTAMP", 
    from_unixtime("TIMESTAMP").alias("HUMAN_READABLE_TIME")
).show(5, truncate=False)
print("go")

GENERAL DATA INSIGHTS : 
+-------------------+---------+--------+----------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 2. Geospatial Transformation: Porto to Casablanca

This section builds a custom Spark UDF to remap GPS trajectories from Porto's bounding box to Casablanca's bounding box. The output column (`CASA_POLYLINE`) keeps trajectories in JSON string format so it can be reused directly for map validation and producer export.

In [8]:
#create our own UDF to handle the data switch from Porto to CASABLANCA
import json
import builtins
from pyspark.sql.functions import udf,col
from pyspark.sql.types import StringType
print("1. Defining Bounding Boxes...")
PORTO_LON_MIN, PORTO_LON_MAX = -8.690, -8.550
PORTO_LAT_MIN, PORTO_LAT_MAX =  41.100, 41.190

CASA_LON_MIN,  CASA_LON_MAX  = -7.900, -7.300
CASA_LAT_MIN,  CASA_LAT_MAX  =  33.300, 33.700

def switch_to_casa(polyline_str):
    if not polyline_str or polyline_str == '[]':
        return '[]'
    try:
        #cords
        coords=json.loads(polyline_str)
        new_coords = []
        for lon, lat in coords:
            # CLAMP Porto coordinates to valid bounding box (removes outliers)
            porto_lon_clamped = builtins.max(PORTO_LON_MIN, builtins.min(PORTO_LON_MAX, float(lon)))
            porto_lat_clamped = builtins.max(PORTO_LAT_MIN, builtins.min(PORTO_LAT_MAX, float(lat)))
            
            # Apply the linear transformation formula
            new_lon = (porto_lon_clamped - PORTO_LON_MIN) / (PORTO_LON_MAX - PORTO_LON_MIN) * (CASA_LON_MAX - CASA_LON_MIN) + CASA_LON_MIN
            new_lat = (porto_lat_clamped - PORTO_LAT_MIN) / (PORTO_LAT_MAX - PORTO_LAT_MIN) * (CASA_LAT_MAX - CASA_LAT_MIN) + CASA_LAT_MIN
            
            if new_lon > -7.46:
                return '[]'
            
            # VALIDATE Casablanca output is within bounds (safety check)
            if CASA_LON_MIN <= new_lon <= CASA_LON_MAX and CASA_LAT_MIN <= new_lat <= CASA_LAT_MAX:
                new_coords.append((new_lon, new_lat))
        
        return json.dumps(new_coords) if new_coords else '[]'
    except Exception:
        return '[]'
    
#register Pyspark UDF
teleport_udf=udf(switch_to_casa,StringType()) 
    
df_casa=df_clean.withColumn("CASA_POLYLINE",teleport_udf(col("POLYLINE")))
df_casa.select("TRIP_ID", "POLYLINE", "CASA_POLYLINE").show(5, truncate=50)
print("go")

1. Defining Bounding Boxes...
+-------------------+--------------------------------------------------+--------------------------------------------------+
|            TRIP_ID|                                          POLYLINE|                                     CASA_POLYLINE|
+-------------------+--------------------------------------------------+--------------------------------------------------+
|1372636858620000589|[[-8.618643,41.141412],[-8.618499,41.141376],[-...|[[-7.594184285714287, 33.48405333333334], [-7.5...|
|1372637303620000596|[[-8.639847,41.159826],[-8.640351,41.159871],[-...|[[-7.68505857142857, 33.56589333333335], [-7.68...|
|1372636951620000320|[[-8.612964,41.140359],[-8.613378,41.14035],[-8...|[[-7.569845714285713, 33.47937333333332], [-7.5...|
|1372636854620000520|[[-8.574678,41.151951],[-8.574705,41.151942],[-...|                                                []|
|1372637091620000337|[[-8.645994,41.18049],[-8.645949,41.180517],[-8...|[[-7.711402857142858, 33.65773

## 2.1 Visual Validation with Folium (100 Trips)

This section validates trajectory quality by plotting up to 100 transformed trips on the same Casablanca map. Trips are sampled from non-empty polylines, rendered in multiple colors, and the map auto-fits to show all plotted routes.

In [15]:
!pip install folium osmnx networkx
!pip install scikit-learn
print("go")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


go



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import folium
import json
import osmnx as ox
import networkx as nx
from pyspark.sql.functions import col

print("1. Extracting 100 random sample trips...")
sample_rows = (
    df_clean.filter(col("POLYLINE").isNotNull() & (col("POLYLINE") != "[]"))
            .sample(fraction=0.01, seed=42)
            .select("POLYLINE")
            .limit(100)
            .collect()
)

sample_trips = []
for row in sample_rows:
    casa_polyline = switch_to_casa(row["POLYLINE"])
    if casa_polyline != "[]":
        sample_trips.append(casa_polyline)

print(f"Collected {len(sample_trips)} transformed trips.")

print("2. Downloading Greater Casablanca Street Network (this might take a minute)...")
try:
    G = ox.graph_from_point((33.55, -7.58), dist=25000, custom_filter='["highway"]["area"!~"yes"]["highway"!~"motorway|motorway_link|trunk|trunk_link"]')
    has_graph = True
except Exception as e:
    print(f"Could not load road network: {e}")
    has_graph = False

print("3. Snapping trips to real roads and generating map...")
casa_map = folium.Map(location=[33.55, -7.58], zoom_start=12)

trip_count = 0
for idx, casa_polyline_str in enumerate(sample_trips):
    try:
        casa_coords = json.loads(casa_polyline_str)
        if len(casa_coords) < 2:
            continue
            
        real_route = []
        if has_graph:
            start_lon, start_lat = casa_coords[0]
            end_lon, end_lat = casa_coords[-1]
            orig_node = ox.distance.nearest_nodes(G, start_lon, start_lat)
            dest_node = ox.distance.nearest_nodes(G, end_lon, end_lat)
            try:
                route = nx.shortest_path(G, orig_node, dest_node, weight="length")
                real_route = [[G.nodes[n]['y'], G.nodes[n]['x']] for n in route] # Folium uses lat, lon
            except nx.NetworkXNoPath:
                pass
                
        if len(real_route) > 0:
            folium.PolyLine(
                locations=real_route,
                color='blue',
                weight=3,
                opacity=0.8
            ).add_to(casa_map)
            
            folium.CircleMarker(
                location=real_route[0],
                radius=3,
                color='green',
                fill=True
            ).add_to(casa_map)
            
            trip_count += 1
            
    except Exception as e:
        print(f"Skipped trip {idx} due to error: {e}")

casa_map.save("casablanca_map_100_trips.html")
print(f"4. Map successfully saved with {trip_count} mapped trips! Open 'casablanca_map_100_trips.html'.")
print("go")


1. Extracting 100 random sample trips...
Collected 77 transformed trips.
2. Downloading Greater Casablanca Street Network (this might take a minute)...
3. Snapping trips to real roads and generating map...
Skipped trip 0 due to error: scikit-learn must be installed as an optional dependency to search an unprojected graph.
Skipped trip 1 due to error: scikit-learn must be installed as an optional dependency to search an unprojected graph.
Skipped trip 2 due to error: scikit-learn must be installed as an optional dependency to search an unprojected graph.
Skipped trip 3 due to error: scikit-learn must be installed as an optional dependency to search an unprojected graph.
Skipped trip 4 due to error: scikit-learn must be installed as an optional dependency to search an unprojected graph.
Skipped trip 5 due to error: scikit-learn must be installed as an optional dependency to search an unprojected graph.
Skipped trip 6 due to error: scikit-learn must be installed as an optional dependency 

## 3. Export for Streaming Simulation

After validating transformed trips, this step prepares a bounded export (100K rows) for Kafka simulation. The reduced sample helps keep local CPU and memory usage stable while preserving realistic trajectory behavior.

In [11]:
print("1. Prepping the data for the Kafka Producer...")

#to protect my cpu for now i will take only 100K rows to test on the producer for now 
df_export = df_casa.filter(df_casa.CASA_POLYLINE != '[]') \
                   .select("TRIP_ID", "TAXI_ID", "TIMESTAMP", "CASA_POLYLINE") \
                   .limit(100000)

print("2. Saving to a single CSV file...")

df_export.toPandas().to_csv("casablanca_teleported.csv", index=False)

print("3. Success! 'casablanca_teleported.csv' is now saved in your project folder.")
print("go")

1. Prepping the data for the Kafka Producer...
2. Saving to a single CSV file...
3. Success! 'casablanca_teleported.csv' is now saved in your project folder.
go


In [12]:
print("1. Loading the exported Casablanca CSV...")

df_check = spark.read.option("header", "true") \
                     .option("inferSchema", "true") \
                     .csv("casablanca_teleported.csv")

total_teleported = df_check.count()
print(f"Total rows ready for Kafka: {total_teleported:,}\n")

print("2. Schema Check (Making sure column names are right):")
df_check.printSchema()

print("\n3. Data Sample (Checking the teleported coordinates):")

df_check.show(5, truncate=80)
print("go")

1. Loading the exported Casablanca CSV...
Total rows ready for Kafka: 100,000

2. Schema Check (Making sure column names are right):
root
 |-- TRIP_ID: long (nullable = true)
 |-- TAXI_ID: integer (nullable = true)
 |-- TIMESTAMP: integer (nullable = true)
 |-- CASA_POLYLINE: string (nullable = true)


3. Data Sample (Checking the teleported coordinates):
+-------------------+--------+----------+--------------------------------------------------------------------------------+
|            TRIP_ID| TAXI_ID| TIMESTAMP|                                                                   CASA_POLYLINE|
+-------------------+--------+----------+--------------------------------------------------------------------------------+
|1372636858620000589|20000589|1372636858|[[-7.594184285714287, 33.48405333333334], [-7.593567142857142, 33.48389333333...|
|1372637303620000596|20000596|1372637303|[[-7.68505857142857, 33.56589333333335], [-7.687218571428575, 33.566093333333...|
|1372636951620000320|200003

## 4. NYC Data Pipeline (Batch + Streaming Preparation)

The following cells profile raw NYC parquet data, quantify missing values, and apply strict cleaning rules. The objective is to produce high-quality batch data and align it with the real-time pipeline requirements for Kafka producers and downstream analytics.

In [13]:
from pyspark.sql.functions import col, count, when, isnull

print("1. Loading all 3 months of raw NYC Parquet data...")
df_nyc_raw = spark.read.parquet("newyork/*.parquet")

total_rows = df_nyc_raw.count()
print(f"Total raw rows: {total_rows:,}\n")

print("2. Null Value Scan:")
null_exprs = [count(when(isnull(c), c)).alias(c) for c in df_nyc_raw.columns]
df_nulls = df_nyc_raw.select(*null_exprs).toPandas().T
df_nulls.columns = ["Missing Values"]
df_nulls["Missing Percentage (%)"] = round((df_nulls["Missing Values"] / total_rows) * 100, 2)
display(df_nulls)

print("\n3. Deep Sanity Check on ALL Numeric Columns:")
numeric_cols = [
    "passenger_count", "trip_distance", "fare_amount", 
    "extra", "mta_tax", "tip_amount", "tolls_amount", 
    "improvement_surcharge", "total_amount", "congestion_surcharge"
]
df_nyc_raw.select(*numeric_cols).summary("min", "mean", "max").show()
print("go")

1. Loading all 3 months of raw NYC Parquet data...
Total raw rows: 22,612,607

2. Null Value Scan:


,Missing Values,Missing Percentage (%)
VendorID,0,0.00
tpep_pickup_datetime,0,0.00
tpep_dropoff_datetime,0,0.00
passenger_count,91809,0.41
trip_distance,0,0.00
RatecodeID,91809,0.41
store_and_fwd_flag,91809,0.41
PULocationID,0,0.00
DOLocationID,0,0.00
payment_type,0,0.00



3. Deep Sanity Check on ALL Numeric Columns:
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+-------------------+---------------------+------------------+--------------------+
|summary|   passenger_count|     trip_distance|       fare_amount|            extra|           mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+-------------------+---------------------+------------------+--------------------+
|    min|               0.0|               0.0|            -447.0|            -60.0|              -0.5|            -89.89|              -70.0|                 -0.3|            -450.3|                -2.5|
|   mean|1.5711894844934002|2.9244857680497747|12.908146671018459|0.898182593011055|0.4956150942702007|2.0645303471708996|0.3480039824

## 4. NYC Batch Dataset Hardening

This final stage applies strict data quality constraints to NYC records and writes a clean parquet dataset to `newyork/processed/nyc_clean.parquet`. These filters remove outliers and invalid records so downstream analytics and ML features are more reliable.

In [14]:
from pyspark.sql.functions import col

print("1. Loading all 3 months of raw NYC Parquet data...")
df_nyc_raw = spark.read.parquet("newyork/*.parquet")
total_rows = df_nyc_raw.count()

print("2. Applying the STRICT ML-focused EDA fixes...")

df_nyc_clean = df_nyc_raw.drop("airport_fee")
df_nyc_clean = df_nyc_clean.fillna({"congestion_surcharge": 0.0})


df_nyc_clean = df_nyc_clean.filter(
    (col("passenger_count") >= 1) & (col("passenger_count") <= 6) & # 0 is a ghost, >6 is illegal
    (col("trip_distance") > 0.0) & (col("trip_distance") < 100.0) & # Capping at 100 miles
    (col("fare_amount") > 0.0) & (col("fare_amount") < 500.0) &     # Crushing the $943k error
    (col("extra") >= 0.0) &                                         # No negative extras
    (col("mta_tax") >= 0.0) &                                       # No negative taxes
    (col("tip_amount") >= 0.0) & (col("tip_amount") < 200.0) &      # Crushing the $141k tip
    (col("tolls_amount") >= 0.0) & (col("tolls_amount") < 100.0) &  # Crushing the $3k toll
    (col("improvement_surcharge") >= 0.0) &                         # No negative surcharges
    (col("total_amount") > 0.0) & (col("total_amount") < 600.0) &   # Capping the total sum
    (col("congestion_surcharge") >= 0.0) &                          # Removing negative fees
    (col("tpep_pickup_datetime") >= '2019-01-01') &                 # Must be in 2019
    (col("tpep_pickup_datetime") < '2019-04-01')                    # Must be before April
).dropna(subset=["PULocationID", "DOLocationID"])                   # Drop missing locations

clean_count = df_nyc_clean.count()
print(f"Total clean rows: {clean_count:,}")
print(f"Rows dropped: {total_rows - clean_count:,} garbage records removed.")


print("\n3. Saving the pristine dataset to Parquet...")
df_nyc_clean.write.mode("overwrite").parquet("newyork/processed/nyc_clean.parquet")
print("Success! The Batch data is 100% locked in and ML-ready.")
print("go")

1. Loading all 3 months of raw NYC Parquet data...
2. Applying the STRICT ML-focused EDA fixes...
Total clean rows: 21,963,501
Rows dropped: 649,106 garbage records removed.

3. Saving the pristine dataset to Parquet...
Success! The Batch data is 100% locked in and ML-ready.
go
